In [ ]:
# === ARRANQUE EN COLAB: árbol de carpetas de la sesión =====================
# Este cuaderno se escribió para correr desde la carpeta `notebook/` de su
# sesión, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese árbol, así que aquí se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E01_fundamentos"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesion EPE E1 - Fundamentos: CRISP-DM + exploracion de datos + A/B testing

**Curso "Herramientas de Ciencias de Datos" - Modalidad EPE - UPC - Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E01_fundamentos/notebook/EPE_S1_fundamentos.ipynb)

> Enfoque EPE: se prioriza la **intuicion y la decision de negocio** sobre el
> formalismo matematico. Las pruebas estadisticas se usan como **herramienta**,
> no se derivan. Este cuaderno recorre **CRISP-DM (roadmap del proyecto)** y
> **EDA + inferencia + A/B testing** al nivel de intuicion y decision de negocio.

## Objetivos de aprendizaje
Al terminar la sesion, el participante es capaz de:
1. **Enmarcar** un problema de negocio con la metodologia **CRISP-DM**.
2. **Explorar** un conjunto de datos con estadistica descriptiva y visualizaciones claras (tipos de variable, faltantes, atipicos).
3. **Comparar dos grupos** con una prueba de hipotesis sencilla (t / chi-cuadrado) y leer el **p-valor** sin tecnicismos.
4. **Disenar e interpretar** un experimento **A/B** para decidir, distinguiendo lo **estadisticamente significativo** de lo **relevante para el negocio**.

## Mapa de la sesion
| # | Bloque | Datos |
|---|---|---|
| a | CRISP-DM como hoja de ruta de un proyecto de datos | -- |
| b | Exploracion de datos (EDA) | Mall Customers (200 clientes) |
| c | Comparar dos grupos sin tecnicismos (hipotesis, p-valor, t/chi2) | Mall Customers |
| d | A/B testing como herramienta de decision | Cookie Cats (retencion) |
| e | Cierre: recomendacion de negocio | -- |

**Materiales hermanos de esta sesion:** guia de laboratorio `laboratorio/GUIA_LABORATORIO_E01.docx`,
plantillas `plantillas/plantilla_exploracion.docx` y `plantillas/guia_ab.docx`, ficha del proyecto
integrador `plantillas/ficha_proyecto_crispdm.docx`, ejercicios `evaluacion/drills.docx`,
entregable evaluable `evaluacion/entregable.docx` y fuentes de actualidad las fuentes de actualidad de la sesión.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el núcleo científico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SÍ. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aquí solo se instala lo que Colab NO trae.
import sys

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los últimos decimales; el método y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "seaborn": "0.13.2",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerías de la sesión
import os, sys, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Estética (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_INK, UPC_GRAY = "#E4002B", "#2D2D2D", "#9AA0A6"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Librerías OK -", "Colab" if "google.colab" in sys.modules else "entorno local")

**🔎 Cargar los dos casos de negocio.** La siguiente celda trae los datos de los dos casos de la sesión: **Mall Customers** (200 clientes, para explorar) y **Cookie Cats** (retención de un juego, para el A/B). Es la fase *Comprensión de los datos* de CRISP-DM: sin datos a la vista, no hay análisis ni decisión.

In [ ]:
# === DATOS DE LA SESIÓN EMBEBIDOS (para Google Colab) ======================
# En Colab no existe la carpeta data/ del curso. Los CSV de la sesión viajan
# comprimidos DENTRO de este cuaderno y se escriben en el directorio de trabajo
# antes de cargarse: son los MISMOS bytes que en local (mismo SHA256), de modo
# que las cifras de la sesión no cambian. En local esta celda no hace nada.
import base64, gzip, os, sys

_EMBEBIDOS = {
    "Mall_Customers.csv":  # 3,981 B  sha256=416a4f62a8f33841...
        "H4sIAGA7mWoC/11Xza6VNwzcH+m8w110UaSvUpz/LFGrVixY8QSIHiFUuFQXeP/OOIljujua4zhjexz7+/3Ht+9fvzxe3vxx/fV4/vvxcr3++LhePz//eP/56c3zB/z39Os/v7y63v2Lvz89f3x69+HrCzD5TUJ4db/J9fb958cl45JypXG/xQlEIdDlfkvXn48vCoVL6lXvt2xIItLa/VY2lHCuXTncb9WsIqGGg82syiWdrrp31a+R77cxCdRMUgkMg50KhFoEtmjXpsxxSqL3Pa6BUMSol67sC7BDPhMjeQSq3lJTK95ZVxai2tCX547kJPrqyyoQqQhHFvkSGVBkOoM/l67RgS32RLJ6ijvrijQwiMY9V8I0yuuYEFAjS3tBMF0TETf1QaQjWdGo50IsEeuHAgqBMsN+WwUeTix9cOUB1pGstFMfWA9cmCzzTEtQYimtPKQLP2m0uCNlAAYYJOOeBzFyT9W7SlN+h3284EbIoTtJpJmwZPzhhZWkUIPTLrCGLGbZWIXgu6Y/7/RnImSXVwAZBqBHX9lJHBhzln+KgQkHVp3AaEf/FkMJTG8hdqTPfrky4soWA5JNDExKcPduTNy9IF+IrRiY74jqAEn+ZIJGgK06lEFErYrnhsPgVpaGwJsIIi0nAvjKsAW2NFQHkYwcleFjqrDFMxBMCkBoVZeCWlAbItFeISJgUI07Whw0GGHNTu3E0F51tS7skVC9r5rSiNDmKGha8XGy7Fe1y6hcHYfFPNlO7qNaoUbNsR/KtVkDMF9DK9QO/6CJpbf8M8bIW7F6aFBAdv9WIqxZ8/1LDGpvK/cUU1b2bbhepRVY9OB0Qjtiiz8qSitkrG/ldEUQUU/2vPIUbbJ7joiBVy+uW3kS7Hu1vilTE/0op+il4Nq7mw3EeOfp38z3mtocwWkTGP0NcZUrswIjWs0pbPAdp3/rzP7I7n0jRv/F1bfO3hrV3gKWiUhz9a2zU4fvXi0nsHHuFPWFEWsTghDnSBBXJwifbCVEVyiCneCpAqCih7OJgxBnUFgKmgepKQn+GaU3pWJRoEPQLmQsYckItWIH6eE9f7tCHIayopgHlZsN4KK+dCRbDGigOp8bOSMY6qpTgSLZGq1OkYhYJbgXlHVJ9R7LpHwGMe6gpTLsPn9lJkuGe7P4jNAyeklR35z/eyCzlm3mMEZPfG404oZy4LNHLYuN5UFosJxnLuNEC7oISKzuOW5r88CNb9fL1NZSEU9rAxoKDXslAAmvSMGaFhAHm+zJDJXyIJFo1MzorHPYIThRCfrZBrDpDcW0i9dMz1Y3noB1hu/mMwfxxajS3i6iIsx7Oo8r3kFhNrNfjQDqYnfGc+G81nv3eKa7cg26yycMJKDO8rgRHQlyXZFcbLXCQyp6x27xTmioVbPeBcR5L2dA0xvIMFPZv1J4hCP9lV0LhQapFDl17bqzSHHF6DN3Z0Ln6U2PZqe81nVxljOlWV14VLC6R4g+WfPSbJmlHRG/LdGhXj08Z5rV8D8z3YNPObDoj6nQGr0OsBaqZbICY8SomVcVMC3cHtjIGeYHlzSp1W1uAFWndRckXpgpmpgzs7ka1tka1a99mDWRqW7BBAOo8t49tiE6QLrjt2gy6HM3lLaCmL5UpS17iLuhtLMy4YK+vFWXPoCdobUVRelqp1d0e2sAVf20OO095iIsPdh+Dkgl1a0UiaLVNV56dCIFqLfu8Q2dj3Xrmd8o2VjyOwMcXTeGfv7JnuCIApA2hxvhbF18fijFbukjVtXjcB45wFT5Z4wjiUQ7KZ1Bzq0eaGQahw+IqGZoj3N+J2Ck8Krh24SgvrzD94noFyvR6j8O9xfd8HsVn+eofs/bRUy7fk92Yki+8Btuj3Z+lAJjD/wHx+ykW40PAAA=",
    "cookie_cats_muestra.csv":  # 93,119 B  sha256=83ed79d2d30477f1...
        "...",
}

def _escribir_embebidos(destinos=None):
    """Vuelca los CSV embebidos al directorio actual y a ../data: los cuadernos
    del curso leen de uno o de otro según cómo calculen su carpeta de datos."""
    if destinos is None:
        destinos = [os.getcwd(), os.path.join(os.getcwd(), "data"),
                    os.path.join(os.path.dirname(os.getcwd()), "data")]
    escritos = []
    for _dest in destinos:
        try:
            os.makedirs(_dest, exist_ok=True)
        except OSError:
            continue
        for _nombre, _b64 in _EMBEBIDOS.items():
            _ruta = os.path.join(_dest, _nombre)
            if not os.path.exists(_ruta):
                with open(_ruta, "wb") as _fh:
                    _fh.write(gzip.decompress(base64.b64decode(_b64)))
                escritos.append(_nombre)
    return sorted(set(escritos))

if "google.colab" in sys.modules:
    _e = _escribir_embebidos()
    print("Datos de la sesión listos en Colab:", ", ".join(_e) if _e else "ya estaban")


In [ ]:
# Rutas robustas (nbconvert local y Colab) + CARGA ENDURECIDA de datos.
# Espeja data/descargar_datos.py: multi-mirror con fallback + checksum SHA256 de la
# base completa + verificación de esquema (columnas y n de filas), para que Colab
# reproduzca lo mismo que local sin deriva silenciosa. El 1.er espejo de cada archivo
# es el repo del curso (bytes versionados); el resto son mirrors abiertos de respaldo.
import urllib.request, hashlib

def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
if "google.colab" in sys.modules:
    DATA = os.getcwd()
else:
    DATA = os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E01_resultados.xlsx")

# Espejos por archivo (se prueban en orden). El PRIMERO es la carpeta de datos que
# ya viaja junto al cuaderno en Drive (LEEME.md de la carpeta de Colab): se descarga
# por su file id, sin montar Drive ni pedir permisos. El repo del curso y los mirrors
# abiertos quedan como respaldo si esa carpeta se mueve o el enlace cambia.
def _drive(file_id):
    return f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

DRIVE_ID = {
    "Mall_Customers.csv": "1Ibzj6qAm9EQ9ImYF_8e2n-M_eqq9dx_y",
    "cookie_cats.csv": "10c0LibUWs5R5RE7oFPkfq2oKk4C8Gwfn",
    "cookie_cats_muestra.csv": "1M144b-na18bToaMsbwdek0fS8JjILNy5",
}
_REPO = ("https://raw.githubusercontent.com/jonatanfigueroagil-creator/"
         "Herramientas-de-Ciencias-de-Datos/master/Sesiones_EPE/E01_fundamentos/data/")
MIRRORS = {
    "Mall_Customers.csv": [
        _drive(DRIVE_ID["Mall_Customers.csv"]),
        _REPO + "Mall_Customers.csv",
        ("https://raw.githubusercontent.com/tirthajyoti/"
         "Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv"),
    ],
    "cookie_cats.csv": [
        _drive(DRIVE_ID["cookie_cats.csv"]),
        _REPO + "cookie_cats.csv",
        ("https://raw.githubusercontent.com/ryanschaub/"
         "Mobile-Games-A-B-Testing-with-Cookie-Cats/master/cookie_cats.csv"),
    ],
    "cookie_cats_muestra.csv": [
        _drive(DRIVE_ID["cookie_cats_muestra.csv"]),
        _REPO + "cookie_cats_muestra.csv",
    ],
}
# Checksum de la base completa (verificado 06/08/2026, = data/descargar_datos.py).
SHA256_COOKIE = "5ab54d761fbddcd50de7b88e4eaf7837cba4569474f50c043a4d17ee342c46bd"
_COOKIE_COLS = ["userid", "version", "sum_gamerounds", "retention_1", "retention_7"]
ESQUEMAS = {  # (columnas esperadas, n de filas esperado)
    "Mall_Customers.csv": (
        ["CustomerID", "Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"], 200),
    "cookie_cats.csv": (_COOKIE_COLS, 90189),
    "cookie_cats_muestra.csv": (_COOKIE_COLS, 3000),
}

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _norm(df):  # normaliza la columna de sexo si el mirror la trae como 'Genre'
    if "Genre" in df.columns and "Gender" not in df.columns:
        df = df.rename(columns={"Genre": "Gender"})
    return df

def _verificar(df, nombre):
    cols, nrows = ESQUEMAS[nombre]
    faltan = [c for c in cols if c not in df.columns]
    if faltan:
        raise ValueError(f"[{nombre}] faltan columnas {faltan}; hay {list(df.columns)}")
    if len(df) != nrows:
        raise ValueError(f"[{nombre}] se esperaban {nrows} filas y hay {len(df)}")

def asegurar(nombre):
    """Trae 'nombre' a DATA verificado (checksum/esquema). Idempotente.
    Devuelve la ruta local, o None si no se pudo obtener la base completa."""
    ruta = os.path.join(DATA, nombre)
    if os.path.exists(ruta):  # idempotencia: no re-descarga si el local es valido
        try:
            if nombre == "cookie_cats.csv":
                if _sha256(ruta) == SHA256_COOKIE:
                    return ruta
                raise ValueError("checksum local no coincide")
            _verificar(_norm(pd.read_csv(ruta)), nombre)
            return ruta
        except Exception as e:
            print(f"  ! {nombre} local inválido ({e}); se re-obtiene.")
    ultimo = None
    for url in MIRRORS[nombre]:
        try:
            print(f"Descargando {nombre} de {url.split('/')[2]}...")
            raw = urllib.request.urlopen(url, timeout=60).read()
            df = _norm(pd.read_csv(io.BytesIO(raw)))
            if nombre == "cookie_cats_muestra.csv" and len(df) != 3000:
                df = df.sample(n=3000, random_state=42).sort_index()  # mirror completo -> muestra
            _verificar(df, nombre)
            if nombre == "cookie_cats.csv":  # ademas, byte-identidad por checksum
                if hashlib.sha256(raw).hexdigest() != SHA256_COOKIE:
                    raise ValueError("checksum del mirror no coincide")
                with open(ruta, "wb") as f:
                    f.write(raw)
            else:
                df.to_csv(ruta, index=False, encoding="utf-8")
            print(f"  + verificado y guardado -> {os.path.basename(ruta)}")
            return ruta
        except Exception as e:
            ultimo = e
            print(f"  ! mirror falló ({e!r})")
    if nombre == "cookie_cats.csv":  # fallback: la base completa se degrada a la muestra
        print("  !! sin base completa desde ningún espejo; se usará la MUESTRA versionada.")
        return None
    raise RuntimeError(f"No se pudo obtener {nombre} de ningún espejo. Último: {ultimo!r}")

mall = _norm(pd.read_csv(asegurar("Mall_Customers.csv")))
cc_muestra = pd.read_csv(asegurar("cookie_cats_muestra.csv"))
_ruta_cc = asegurar("cookie_cats.csv")
if _ruta_cc is None:            # degradación honesta: el A/B correría sobre la muestra
    cc = cc_muestra.copy()
    USANDO_MUESTRA_COOKIE = True
    print("\n*** AVISO: sin base completa; las cifras estrella del A/B NO reproducen "
          "hasta restaurar cookie_cats.csv (90 189 filas). ***")
else:
    cc = pd.read_csv(_ruta_cc)
    USANDO_MUESTRA_COOKIE = False

print("Mall Customers :", mall.shape)
print("Cookie muestra :", cc_muestra.shape)
print("Cookie completo:", cc.shape)

---
## a) CRISP-DM: la hoja de ruta de un proyecto de datos en la empresa

**CRISP-DM** (*Cross-Industry Standard Process for Data Mining*) organiza cualquier
proyecto de datos en **seis fases** que se recorren de forma **iterativa** (volver
atrás es parte del método, no un error). Es el mapa que ordena todo el curso:
primero se entiende el negocio y recién al final se despliega una decisión.

| # | Fase CRISP-DM | Pregunta que responde | Ejemplo (retención de un juego) |
|---|---|---|---|
| 1 | Comprensión del negocio | Qué decisión se quiere habilitar? | Conviene mover la "puerta" del nivel 30 al 40? |
| 2 | Comprensión de los datos | Qué datos hay y cómo se presentan? | Retención día 1 y día 7 de 90 000 jugadores |
| 3 | Preparación de los datos | Cómo se limpian y ordenan? | Tipos, faltantes, atípicos, grupos del test |
| 4 | Modelado / análisis | Qué técnica se aplica? | Comparar la retención entre control y tratamiento |
| 5 | Evaluación | El resultado sirve para decidir? | El efecto es significativo y relevante? |
| 6 | Despliegue | Cómo se lleva a la acción? | Recomendación: mover o no la puerta |

**Idea clave (intuición, sin fórmulas):** la mayor parte del esfuerzo real de un
proyecto -cerca del **80%**- se concentra en **entender y preparar los datos**
(fases 2 y 3), no en el modelado. Un buen EDA (bloque *b*) es, por eso, la mitad
del trabajo. La *Ficha de proyecto CRISP-DM* (`plantillas/ficha_proyecto_crispdm.docx`)
es el punto de partida del **proyecto integrador** del curso.

**🔎 CRISP-DM, en una tabla que se ejecuta.** La celda imprime las **seis fases** y en qué bloque de la sesión cae cada una. Lo que importa al negocio: el proyecto **empieza y termina en el negocio** (fases 1 y 6), y volver atrás -de la evaluación a la pregunta- es parte del método, no un fallo.

In [ ]:
# Las 6 fases de CRISP-DM como marco de referencia de la sesión.
crispdm = pd.DataFrame({
    "fase": [1, 2, 3, 4, 5, 6],
    "nombre": ["Comprensión del negocio", "Comprensión de los datos",
               "Preparación de los datos", "Modelado / análisis",
               "Evaluación", "Despliegue"],
    "bloque_de_la_sesion": ["a", "b", "b", "c y d", "c y d", "e"],
})
print(crispdm.to_string(index=False))
print("\nCRISP-DM es iterativo: la evaluación (5) suele mandar de vuelta a las fases 1-3.")

---
## b) Exploracion de datos (EDA): Mall Customers

**Caso de negocio.** Un centro comercial quiere **entender a sus clientes** para
orientar campanas. Dispone de una tabla con **200 clientes**: sexo, edad, ingreso
anual (en miles de dolares) y un **puntaje de gasto** (1-100) que el mall asigna
segun el comportamiento de compra. El EDA responde: *como son estos clientes y
que relaciones hay entre sus caracteristicas?*

El **analisis exploratorio de datos (EDA)** es mirar los datos -sobre todo de
forma grafica- **antes** de modelar, para descubrir su estructura, sus patrones y
sus problemas de calidad.

**🔎 Primer vistazo antes de calcular nada.** La celda muestra **cuántas filas y columnas** hay y de **qué tipo** es cada variable. El tipo manda: decide qué estadístico, gráfico y prueba tienen sentido (una categoría se cuenta, no se promedia).

In [ ]:
# Primer vistazo: forma, tipos de variable y primeras filas.
print("Dimensiones (filas, columnas):", mall.shape, "\n")
print("Tipos de variable:")
print(mall.dtypes, "\n")
mall.head()

**Tipos de variable (por qué importan).** El tipo decide qué estadístico, gráfico
y prueba son válidos:
- **`Gender`**: cualitativa **nominal** (categoría sin orden) -> se cuenta y se
  grafica con barras; nunca se promedia.
- **`Age`, `Annual Income (k$)`, `Spending Score (1-100)`**: cuantitativas ->
  admiten media, histograma, correlación.
- **`CustomerID`**: identificador, **no** es una variable de análisis.

In [ ]:
# Estadística descriptiva de las variables numéricas.
num = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
desc = mall[num].describe().round(2)
desc

**📖 Cómo se lee la tabla descriptiva.** Cada columna resume una variable: **media** (valor típico), **std** (dispersión), **min**/**max** y los **cuartiles** (25%, 50% = mediana, 75%). Si la media se aleja de la mediana, la variable es asimétrica; una brecha grande entre el 75% y el máximo anticipa posibles **atípicos**, justo lo que revisa la celda siguiente.

**⚠️ Riesgo 1 — GIGO: *garbage in, garbage out*.** Ninguna recomendación es mejor que los datos que la sostienen. Por eso, antes de analizar, se cuentan **faltantes** y **atípicos**: un solo valor extremo (un ingreso mal digitado) puede distorsionar un promedio y sesgar la decisión. Qué hacer con un atípico -corregir, eliminar o conservar- es una **decisión de negocio**, no solo técnica.

In [ ]:
# Calidad de datos: valores faltantes y atípicos (regla del rango intercuartílico).
faltantes = mall.isna().sum()
print("Valores faltantes por columna:")
print(faltantes.to_string(), "\n")

filas_atipicos = []
for col in num:
    q1, q3 = mall[col].quantile([0.25, 0.75]); ric = q3 - q1
    lo, hi = q1 - 1.5 * ric, q3 + 1.5 * ric
    n_out = int(((mall[col] < lo) | (mall[col] > hi)).sum())
    filas_atipicos.append({"variable": col, "limite_inf": round(lo, 1),
                           "limite_sup": round(hi, 1), "n_atipicos": n_out})
atipicos = pd.DataFrame(filas_atipicos)
print("Atípicos por variable (regla 1.5 x RIC):")
print(atipicos.to_string(index=False))

**Lectura de calidad de datos.** No hay **valores faltantes** (0 en todas las
columnas), así que no hace falta imputar. Solo aparecen **2 atípicos** en el
ingreso anual (los dos clientes de mayor renta, ~137 mil): no son errores, son
clientes reales de alto poder adquisitivo, por lo que **se conservan**. Decidir
qué hacer con un atípico -corregir, eliminar o conservar- es una decisión de
**negocio**, no solo técnica.

**🔎 Cinco gráficos, cinco preguntas de negocio.** Las tres celdas siguientes generan los cinco gráficos del EDA: **histograma** (cómo se distribuye una variable?), **barras** (cuántos por categoría?), **boxplot** (difiere un grupo de otro y dónde están los atípicos?), **dispersión** (dos variables se mueven juntas?) y **mapa de correlación** (cuáles se asocian?). El gráfico se elige según el **tipo de variable**.

In [ ]:
# --- Visualizaciones EDA (se trazan directo de los datos crudos) ---
# 1) Distribuciones (histograma) de las tres variables numéricas.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, num):
    sns.histplot(mall[col], bins=15, color=UPC_RED, alpha=0.75, ax=ax)
    ax.set_title(col); ax.set_ylabel("N clientes")
fig.suptitle("Distribución de las variables numéricas (Mall Customers)", y=1.04, fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "eda_mall_distribuciones.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 2) Barras (conteo por sexo) y 3) boxplot (gasto por sexo).
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.countplot(data=mall, x="Gender", hue="Gender", palette=[UPC_RED, UPC_GRAY],
              legend=False, ax=axes[0])
axes[0].set_title("Clientes por sexo"); axes[0].set_ylabel("N clientes"); axes[0].set_xlabel("")
sns.boxplot(data=mall, x="Gender", y="Spending Score (1-100)", hue="Gender",
            palette=[UPC_RED, UPC_GRAY], legend=False, ax=axes[1])
axes[1].set_title("Puntaje de gasto por sexo"); axes[1].set_xlabel("")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "eda_mall_barras_boxplot.png"), bbox_inches="tight")
plt.show()

**❓ Qué se quiere averiguar.** ¿Los clientes del mall forman grupos con comportamientos distintos, o son un conjunto homogéneo al que conviene dirigir un mismo mensaje?

- **Qué decide:** si aparecen grupos, el mall tiene una razón para partir su presupuesto de marketing; si la nube es homogénea, una sola campaña general es lo eficiente.
- **Antes de mirar el resultado:** si la dispersión ingreso vs gasto resulta **sin estructura**, no hay a quién separar. Si se ven **agrupaciones**, hay base para una segmentación. Y en el mapa de correlación: una correlación **cercana a 0** entre ingreso y gasto significaría que el nivel de ingreso del cliente no anticipa lo que gasta en el mall, un resultado incómodo para quien planifica por nivel socioeconómico.

**⚠️ Riesgo 2 — correlación no es causalidad.** La dispersión y el mapa de correlación de la celda siguiente muestran **asociación**: que variables se mueven juntas. Que se muevan juntas **no prueba** que una cause la otra (puede haber una tercera variable detrás). Para afirmar causa hace falta un **experimento**, el A/B del bloque *d*.

In [ ]:
# 4) Dispersión (ingreso vs gasto) y 5) mapa de correlación.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.scatterplot(data=mall, x="Annual Income (k$)", y="Spending Score (1-100)",
                hue="Gender", palette=[UPC_RED, UPC_GRAY], s=45, ax=axes[0])
axes[0].set_title("Ingreso anual vs puntaje de gasto")

corr = mall[num].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdGy_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.7}, ax=axes[1])
axes[1].set_title("Mapa de correlación")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "eda_mall_dispersion_correlacion.png"), bbox_inches="tight")
plt.show()
print("Correlaciones:\n", corr.round(3).to_string())

**Lectura de negocio del EDA.**
- La **dispersión** ingreso vs gasto muestra **agrupaciones visibles** de clientes
  (por ejemplo, alto ingreso con gasto alto y alto ingreso con gasto bajo). Es la
  semilla de una **segmentación** de clientes -técnica que se profundiza más
  adelante en el curso-; aquí solo se observa.
- En el **mapa de correlación**, la edad y el puntaje de gasto tienen una
  correlación **negativa moderada (~ -0.33)**: los clientes **más jóvenes tienden a
  gastar más**. En cambio, ingreso y gasto casi no se mueven juntos (~0.01): tener
  más ingreso **no** implica mayor puntaje de gasto en este mall.
- **Aviso clave:** la correlación **explora asociación, no causalidad**. Que dos
  variables se muevan juntas no prueba que una cause la otra; para afirmar causa se
  necesita un **experimento** (bloque *d*).

---
## c) Comparar dos grupos sin tecnicismos: hipótesis, p-valor, t y chi-cuadrado

A veces una diferencia **se ve** en el gráfico, pero *se ve* no es *es*: podría ser
**azar** de la muestra. Una **prueba de hipótesis** decide si la diferencia
observada es creíble o si podría explicarse solo por casualidad. La idea, sin
fórmulas:

- **Hipótesis nula (H0):** "no hay diferencia" (el punto de partida escéptico).
- **Hipótesis alternativa (H1):** "sí hay diferencia" (lo que se quiere mostrar).
- **p-valor:** *si realmente no hubiera diferencia*, qué tan probable sería ver un
  resultado como el observado (o más extremo). **Pequeño (< 0.05)** = los datos son
  poco compatibles con "no hay diferencia" -> se declara la diferencia. El p-valor
  **no** es la probabilidad de que H1 sea cierta ni el tamaño del efecto.

Qué prueba usar (regla simple):
- **Prueba t** -> comparar el **promedio** de una variable numérica entre **dos grupos**.
- **Prueba chi-cuadrado** -> ver si **dos variables categóricas** están asociadas (tablas de conteos).

**📖 De dónde viene la prueba t (Student, 1908).** La prueba que se usa más abajo tiene autor y una
pregunta detrás. William Sealy Gosset, jefe de experimentación de la cervecería Guinness, firmaba
"Student" porque la empresa prohibía a sus empleados publicar con su nombre, y su artículo de 1908
**no** se propuso comparar dos tratamientos: se propuso "determinar el punto a partir del cual
podemos usar las tablas de la integral de probabilidad para juzgar la significación de la media de
una serie de experimentos, y proporcionar tablas alternativas para cuando el número de experimentos
es demasiado escaso"; es decir, **donde termina la muestra grande y empieza la pequeña**. Eligió ese
método porque el corriente -suponer una normal con desviación s/raíz(n)- falla con pocos datos: la
desviación calculada sobre la propia muestra arrastra su propio error y la curva normal da entonces
"una falsa sensación de seguridad". Su respuesta consiste en dividir por la desviación de la
**muestra** y aceptar que ese denominador también es aleatorio, de donde se derivan las **colas más
gruesas** de la t. Y la decisión que sus tablas habilitaban -si una serie de experimentos ya alcanza
la exactitud requerida o si conviene continuar la investigación- es la misma del bloque *d*: cerrar
un A/B o prolongarlo. *(Student, 1908, "The Probable Error of a Mean", Biometrika 6(1), 1-25. La
variante que ejecuta este cuaderno es la **t de Welch**, un refinamiento posterior que no supone
varianzas iguales entre los dos grupos.)*

**❓ Qué se quiere averiguar.** ¿La hipótesis que sugirió el EDA —que los más jóvenes gastan más— resiste una prueba formal, o es un patrón observado en 200 clientes por simple azar del muestreo?

- **Qué decide:** si la diferencia es real, el mall tiene fundamento para campañas separadas por edad; si es ruido, ese presupuesto diferenciado se gasta sobre una ilusión.
- **Antes de mirar el resultado:** con **p < 0,05** la brecha de puntaje entre menores de 40 y el resto se toma como real; con **p alto** la hipótesis queda sin respaldo. Conviene además decidir de antemano cuántos puntos del índice 1-100 valdría la pena perseguir: una diferencia puede ser significativa y aun así no cubrir el costo de la campaña.

**🔎 Poner la hipótesis a prueba.** El EDA insinuó que los más jóvenes gastan más. La celda parte a los clientes en **menores de 40** y **40 o más** y usa una **prueba t** para decidir si esa diferencia de gasto es real o simple azar del muestreo. El resultado se lee para el negocio: si la diferencia es real, hay margen para campañas diferenciadas por edad.

In [ ]:
# Ejemplo 1 - Prueba t: el puntaje de gasto difiere entre clientes jóvenes y mayores?
# (El EDA sugirió que la edad se asocia negativamente con el gasto; lo ponemos a prueba.)
mall["grupo_edad"] = np.where(mall["Age"] < 40, "Menores de 40", "40 o más")
jov = mall.loc[mall.grupo_edad == "Menores de 40", "Spending Score (1-100)"]
may = mall.loc[mall.grupo_edad == "40 o más", "Spending Score (1-100)"]

t_stat, p_t = stats.ttest_ind(jov, may, equal_var=False)  # Welch (varianzas desiguales)
print(f"Gasto medio  <40 años : {jov.mean():.2f}  (n={len(jov)})")
print(f"Gasto medio  >=40 años: {may.mean():.2f}  (n={len(may)})")
print(f"Diferencia            : {jov.mean() - may.mean():.2f} puntos")
print(f"Prueba t (Welch): t = {t_stat:.2f} ,  p = {p_t:.6f}")
comparacion_t = {"grupo_A": "Menores de 40", "media_A": round(jov.mean(), 2),
                 "grupo_B": "40 o más", "media_B": round(may.mean(), 2),
                 "diferencia": round(jov.mean() - may.mean(), 2),
                 "t_welch": round(t_stat, 3), "p_valor": round(p_t, 6)}

**Lectura (prueba t).** Los clientes menores de 40 registran un **puntaje de gasto**
medio de **~60 puntos** frente a **~37** de los de 40 o más -es el **índice 1-100**
que asigna el mall, **no** un monto en dinero-: una diferencia de **~23 puntos del
índice**. Con **p < 0.001** (muy por debajo de 0.05), esa diferencia **no es
casualidad**: es una diferencia real de comportamiento. *Decisión de negocio:* el
mall tiene margen para diseñar campañas diferenciadas por edad, porque el segmento
joven muestra mayor propensión a gastar.

**❓ Qué se quiere averiguar.** ¿El sexo del cliente dice algo sobre cuánto gasta, o es una variable que el mall arrastra en su base sin que aporte a la decisión?

- **Qué decide:** si el sexo discrimina, se justifican dos líneas de comunicación distintas; si no, separar por sexo duplica el costo de la campaña sin ganancia alguna.
- **Antes de mirar el resultado:** con **p < 0,05** en la tabla sexo por nivel de gasto habría asociación. Con **p alto** —y la comparación útil aquí es contra la edad, que sí separó— la variable no aporta: el mismo dato puede ser buen predictor en una dimensión y ruido en otra.

**💡 El p-valor mide sorpresa, no importancia.** Un p-valor pequeño dice que la diferencia sería **poco probable si no existiera**; no dice que sea grande ni que le convenga al negocio. Y la prueba se elige por el **tipo de dato**: la **t** compara promedios (gasto por edad); la **chi-cuadrado** de la celda siguiente ve si **dos categorías** se asocian (sexo y nivel de gasto).

In [ ]:
# Ejemplo 2 - Prueba chi-cuadrado: el sexo está asociado al nivel de gasto?
mall["nivel_gasto"] = pd.cut(mall["Spending Score (1-100)"], bins=[0, 40, 60, 100],
                             labels=["bajo", "medio", "alto"])
tabla = pd.crosstab(mall["Gender"], mall["nivel_gasto"])
print("Tabla de contingencia (conteos):")
print(tabla.to_string(), "\n")
chi2, p_chi, dof, esp = stats.chi2_contingency(tabla)
print(f"Prueba chi-cuadrado: chi2 = {chi2:.3f} ,  gl = {dof} ,  p = {p_chi:.4f}")
comparacion_chi = {"variables": "Gender x nivel_gasto", "chi2": round(chi2, 3),
                   "gl": dof, "p_valor": round(p_chi, 4)}

**Lectura (prueba chi-cuadrado).** El **p-valor es alto (~0.92 >> 0.05)**: **no hay
evidencia** de que el nivel de gasto dependa del sexo del cliente. La ligera
diferencia entre columnas de la tabla es compatible con el azar. *Decisión de
negocio:* segmentar por sexo **no** aporta aquí; la edad (ejemplo 1) sí discrimina.

**Conclusión de este bloque:** la prueba **aporta el criterio objetivo** cuando la inspección visual no basta.
Una diferencia grande (jóvenes vs mayores) resultó real; una diferencia solo aparente
(sexo) resultó ruido. El p-valor dice **si** hay señal; el **tamaño** de la
diferencia dice **cuánto importa**.

**❓ Qué se quiere averiguar.** ¿La conclusión anterior proviene de los datos, o del corte 40/60 que se eligió de forma manual para discretizar el índice de gasto?

- **Qué decide:** de esto depende cuánto vale el hallazgo fuera del cuaderno. Una conclusión que se invierte al mover un umbral elegido por convención no se puede defender ante un comité.
- **Antes de mirar el resultado:** si la t de Welch sobre el índice sin discretizar también da **p alto**, dos caminos distintos coinciden y la conclusión pertenece al dato. Si diera **p < 0,05**, el corte 40/60 habría escondido una diferencia real y el chi-cuadrado anterior quedaría inservible como argumento.

**💡 Justificar el corte del chi-cuadrado y confirmarlo con la prueba directa.** El chi-cuadrado
anterior partió el **índice de gasto** (escala **1-100**, no dinero) en tres niveles de negocio
-bajo (< 40), medio (40-60) y alto (> 60)- para tratarlo como **categoría** y cruzarlo con el sexo.
Ese corte es una **convención de segmentación** del mall, no un dato; para no hacer depender la
conclusión de él, la celda siguiente responde la **misma** pregunta -«¿el sexo discrimina el
gasto?»- con la prueba **directa y más potente**: una **t de Welch** sobre el índice de gasto
**medio** por sexo, sin discretizar la variable.

In [ ]:
# Ejemplo 2b - Prueba t (Welch): el índice de gasto MEDIO difiere entre sexos?
# Prueba DIRECTA de "el sexo discrimina el gasto", sin binning (complementa al chi-cuadrado).
fem = mall.loc[mall["Gender"] == "Female", "Spending Score (1-100)"]
mas = mall.loc[mall["Gender"] == "Male", "Spending Score (1-100)"]
t_sexo, p_sexo = stats.ttest_ind(fem, mas, equal_var=False)  # Welch (varianzas desiguales)
print(f"Índice de gasto medio  Female: {fem.mean():.2f}  (n={len(fem)})")
print(f"Índice de gasto medio  Male  : {mas.mean():.2f}  (n={len(mas)})")
print(f"Diferencia            : {fem.mean() - mas.mean():.2f} puntos del índice")
print(f"Prueba t (Welch): t = {t_sexo:.3f} ,  p = {p_sexo:.4f}  ->  NO significativo")
comparacion_t_sexo = {"variable": "Spending Score por Gender",
                      "media_Female": round(fem.mean(), 2), "media_Male": round(mas.mean(), 2),
                      "diferencia": round(fem.mean() - mas.mean(), 2),
                      "t_welch": round(t_sexo, 3), "p_valor": round(p_sexo, 4)}

**Lectura (t de Welch por sexo).** Las mujeres promedian **~51,5** puntos del índice de gasto y los
hombres **~48,5**: una diferencia de apenas **~3 puntos**, con **t ≈ 0,80** y **p ≈ 0,42 (>> 0,05)**.
La prueba **directa** llega a la misma conclusión que el chi-cuadrado: el **sexo no discrimina** el
gasto. Que las dos pruebas -una sobre la **media** (t), otra sobre la **tabla de conteos**
(chi-cuadrado)- coincidan es la señal de que la conclusión **no** dependía del corte 40/60. *Regla que
queda:* la **t** compara **promedios** de dos grupos; el **chi-cuadrado**, la asociación de **dos
categorías**.

---
## d) A/B testing como herramienta de decision: Cookie Cats

**Caso de negocio.** *Cookie Cats* (Tactile Entertainment) es un juego movil de
puzles. Al avanzar, el jugador se enfrenta a una **"puerta" (gate)** que lo obliga a
esperar o pagar. El equipo evaluo **mover la primera puerta del nivel 30 al 40** y
corrio un **experimento A/B** con usuarios reales asignados al azar:

- **Control `gate_30`:** puerta en el nivel 30 (situacion actual).
- **Tratamiento `gate_40`:** puerta en el nivel 40 (cambio propuesto).
- **Metricas de retencion:** `retention_1` (volvio al dia 1) y `retention_7`
  (volvio al dia 7). La **metrica primaria de decision (OEC)**, declarada de
  antemano, es la **retencion a 7 dias**.

Un **A/B test** es el estandar de oro para decidir si un cambio mejora un KPI: como
los jugadores se reparten **al azar**, los dos grupos solo difieren -en promedio-
en la puerta, y la diferencia de retencion se puede atribuir al cambio (causa).

**❓ Qué se quiere averiguar.** ¿Cuántos jugadores hacen falta para que un experimento sea capaz de detectar el efecto que busca? La celda lo examina en sentido inverso: ejecuta el A/B con solo 3 000 y muestra su alcance.

- **Qué decide:** el tamaño y la duración del experimento se fijan **antes** de lanzarlo. Un test subdimensionado consume semanas de tráfico y devuelve un veredicto vacío.
- **Antes de mirar el resultado:** si con 3 000 jugadores se obtuviera lo mismo que con los 90 189, el tamaño sería un detalle administrativo. Si resulta **no significativo** y además con el **signo del lift inestable**, entonces un «no hay diferencia» leído sobre una muestra pequeña no prueba nada: solo indica que el tamaño no alcanzó para detectarla.

**🔎 Antes del experimento real: por qué el tamaño es determinante.** La celda repite el A/B con una **muestra pequeña (3 000 jugadores)** para verlo de primera mano: con pocos datos la prueba **no detecta** diferencias aunque existan. Es cuestión de **poder estadístico**, no de que no haya efecto.

In [ ]:
# Vistazo con la MUESTRA pequeña (3 000 jugadores): por qué el tamaño importa.
def resumen_retencion(df, metrica):
    g = df.groupby("version")[metrica].agg(exitos="sum", n="count")
    g["tasa"] = g["exitos"] / g["n"]
    return g

peek = resumen_retencion(cc_muestra, "retention_7")
s40, n40 = int(peek.loc["gate_40", "exitos"]), int(peek.loc["gate_40", "n"])
s30, n30 = int(peek.loc["gate_30", "exitos"]), int(peek.loc["gate_30", "n"])
z_p, p_p = proportions_ztest([s40, s30], [n40, n30])
print("MUESTRA (n=3 000) - retención día 7:")
print(peek.round(4).to_string())
print(f"lift gate_40 vs gate_30 = {(peek.loc['gate_40','tasa']/peek.loc['gate_30','tasa']-1)*100:+.2f}%")
print(f"prueba de proporciones: z = {z_p:.3f} , p = {p_p:.3f}  ->  NO significativo")
peek_row = {"fuente": "muestra n=3000", "gate30_tasa": round(peek.loc['gate_30','tasa'],4),
            "gate40_tasa": round(peek.loc['gate_40','tasa'],4), "z": round(z_p,3), "p_valor": round(p_p,3)}

> **Por qué el tamaño de la muestra importa (poder estadístico).** Con solo **3 000**
> jugadores, la prueba **no detecta** diferencia (p alto) e incluso el signo del
> lift es inestable: el experimento está **subdimensionado** ("poco poder"). Un "no
> significativo" con muestra pequeña **no** prueba que no haya efecto -solo que no
> alcanzó para verlo-. Por eso el experimento real se ejecuta sobre **todos** los
> jugadores. Nunca se decide con una porción pequeña ni se detiene el test en cuanto
> "resulta" significativo (*peeking*).

**❓ Qué se quiere averiguar.** ¿Conviene mover la puerta del juego del nivel 30 al 40? Es la pregunta que motivó la sesión entera, y las cuatro cifras de esta tabla son su respuesta.

- **Qué decide:** la retención a 7 días es la métrica primaria, la que sostiene la base de jugadores activos y con ella el ingreso. La recomendación de cierre del cuaderno se deriva de aquí.
- **Antes de mirar el resultado:** si el **intervalo de la diferencia cruza el 0**, no hay evidencia para modificar el diseño. Si queda **enteramente negativo**, el cambio daña la retención y la respuesta es no. Si quedara **enteramente positivo**, faltaría aún preguntar si el lift es lo bastante grande para cubrir el costo del desarrollo: significativo y relevante no son lo mismo.

**🔎 El experimento completo: la base de la decisión.** Ahora sí, sobre los **90 189 jugadores**. La celda calcula, para la retención a día 1 y día 7, la **tasa de cada grupo**, el **lift** (cuánto cambia, en %), el **p-valor** y el **intervalo de confianza** de la diferencia. Son esas cuatro cifras **juntas** -no una sola- las que sostienen la recomendación.

In [ ]:
# Experimento COMPLETO (90 189 jugadores): la base de la decisión.
filas = []
for metrica in ["retention_1", "retention_7"]:
    g = resumen_retencion(cc, metrica)
    s30, n30 = int(g.loc["gate_30", "exitos"]), int(g.loc["gate_30", "n"])
    s40, n40 = int(g.loc["gate_40", "exitos"]), int(g.loc["gate_40", "n"])
    t30, t40 = s30 / n30, s40 / n40
    diff = t40 - t30
    lift = diff / t30 * 100
    z, p = proportions_ztest([s40, s30], [n40, n30])
    se = np.sqrt(t40 * (1 - t40) / n40 + t30 * (1 - t30) / n30)
    ic_lo, ic_hi = diff - 1.96 * se, diff + 1.96 * se
    filas.append({"metrica": metrica, "gate30_tasa": round(t30, 4), "gate40_tasa": round(t40, 4),
                  "diferencia_pp": round(diff * 100, 3), "lift_pct": round(lift, 2),
                  "z": round(z, 3), "p_valor": round(p, 5),
                  "ic95_dif_inf_pp": round(ic_lo * 100, 3), "ic95_dif_sup_pp": round(ic_hi * 100, 3)})
ab = pd.DataFrame(filas)
print("Experimento completo - retención gate_30 (control) vs gate_40 (tratamiento):")
print(ab.to_string(index=False))

**Lectura de negocio del A/B (experimento completo).**
- **Retención día 1:** control **44.8%** vs tratamiento **44.2%** (lift **-1.3%**),
  con **p ~ 0.074** y un intervalo de la diferencia que **cruza 0** -> diferencia
  **no** concluyente en el día 1.
- **Retención día 7 (métrica primaria):** control **19.0%** vs tratamiento
  **18.2%** (lift **-4.3%**), con **p ~ 0.0016** y un intervalo de la diferencia
  **enteramente negativo** (~ -1.3 a -0.3 pp) -> mover la puerta al nivel 40
  **reduce** la retención a 7 días, y la diferencia es **estadísticamente
  significativa**.
- **Significancia estadística y práctica coinciden aquí:** el efecto es real y su
  signo es **desfavorable**; además, perder retención a 7 días afecta directamente
  la base de jugadores activos.

**❓ Qué se quiere averiguar.** ¿Se puede confiar en el resultado que se acaba de leer, o la aleatorización repartió mal a los jugadores y la comparación viene viciada de origen?

- **Qué decide:** si la asignación falló, ninguna cifra del bloque anterior sirve, por pequeño que sea su p-valor. Esta verificación va antes de firmar la recomendación, no después.
- **Antes de mirar el resultado:** si el reparto resulta compatible con el **50/50** esperado (**p > 0,05** en el chi-cuadrado de bondad de ajuste), el experimento supera el control de calidad. Si resulta **p < 0,05** hay desbalance detectable y corresponde buscar su causa —un filtro que descartó usuarios de un brazo, un fallo de asignación— y recién después juzgar si su magnitud alcanza para invertir la conclusión.

**🔎 Verificación de calidad del experimento: ¿los grupos quedaron del mismo tamaño?** Antes de creer en
el resultado conviene verificar que la **aleatorización repartió bien** a los jugadores. Un desbalance
fuerte de tamaños (*Sample Ratio Mismatch*, SRM) es una señal de alarma: si la asignación falló, la
comparación podría estar sesgada. Se contrasta con un chi-cuadrado de bondad de ajuste contra el
reparto **50/50** esperado.

In [ ]:
# Chequeo SRM (Sample Ratio Mismatch): el reparto control/tratamiento vs. el 50/50 esperado.
conteos = cc["version"].value_counts()
n_c, n_t = int(conteos["gate_30"]), int(conteos["gate_40"])
chi_srm, p_srm = stats.chisquare([n_c, n_t])  # H0: reparto 50/50
print(f"gate_30 (control)    : {n_c}")
print(f"gate_40 (tratamiento): {n_t}")
print(f"SRM chi2 = {chi_srm:.3f} ,  p = {p_srm:.4f}")
print("Interpretación:", "reparto compatible con 50/50" if p_srm >= 0.05
      else "desbalance de asignación detectado (p < 0.05): leer el A/B con cautela")
ab_srm = {"gate_30": n_c, "gate_40": n_t, "esperado_50_50": (n_c + n_t) / 2,
          "chi2": round(chi_srm, 3), "p_valor": round(p_srm, 4)}

**Lectura (SRM).** El reparto fue **44 700** (control) vs **45 489** (tratamiento): el chi-cuadrado
de bondad de ajuste da **chi² ≈ 6,90** y **p ≈ 0,009 (< 0,05)**, es decir un desbalance de tamaños
**estadísticamente detectable**. En un experimento real esto obliga a **investigar la causa** -un
sesgo en la asignación, un filtro que descartó usuarios de un brazo- antes de confiar del todo en el
efecto. Aquí el desbalance es leve en magnitud y **no** invierte la conclusión del OEC (retención a 7
días), pero la verificación es parte del método: la aleatorización se **verifica**, no se supone.

**⚠️ Riesgo 3 — significativo no es lo mismo que relevante (y sin *p-hacking*).** Con muestras muy grandes, diferencias mínimas resultan *significativas*: por eso se mira siempre la **magnitud** (lift + intervalo), no solo el p-valor. Y dos reglas de buena práctica: fijar la **métrica primaria y el tamaño de muestra ANTES** de mirar, y **no** detener el test en cuanto *resulta* significativo (*peeking* / *p-hacking*), porque infla los falsos positivos.

In [ ]:
# --- Exportar TODOS los resultados/pruebas a resultados/E01_resultados.xlsx ---
with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
    mall[num].describe().round(3).to_excel(w, sheet_name="eda_descriptivos")
    atipicos.to_excel(w, sheet_name="eda_calidad", index=False)
    pd.DataFrame([comparacion_t]).to_excel(w, sheet_name="comparacion_t", index=False)
    pd.DataFrame([comparacion_chi]).to_excel(w, sheet_name="comparacion_chi", index=False)
    ab.to_excel(w, sheet_name="ab_retencion", index=False)
    pd.DataFrame([peek_row]).to_excel(w, sheet_name="ab_muestra_peek", index=False)
    # Hojas NUEVAS del rework E01 (no alteran el contenido de las anteriores):
    pd.DataFrame([comparacion_t_sexo]).to_excel(w, sheet_name="comparacion_t_sexo", index=False)
    pd.DataFrame([ab_srm]).to_excel(w, sheet_name="ab_srm", index=False)
print("Excel de resultados escrito en:", XLSX)
print("Hojas:", ["eda_descriptivos", "eda_calidad", "comparacion_t", "comparacion_chi",
                 "ab_retencion", "ab_muestra_peek", "comparacion_t_sexo", "ab_srm"])

In [ ]:
# --- Figuras de RESULTADOS: se generan LEYENDO el Excel (convención del curso) ---
comp = pd.read_excel(XLSX, sheet_name="comparacion_t")
abx = pd.read_excel(XLSX, sheet_name="ab_retencion")

# R1) Gasto medio por grupo de edad (resultado de la prueba t).
fig, ax = plt.subplots(figsize=(5.6, 3.8))
vals = [comp.loc[0, "media_A"], comp.loc[0, "media_B"]]
labels = [comp.loc[0, "grupo_A"], comp.loc[0, "grupo_B"]]
bars = ax.bar(labels, vals, color=[UPC_RED, UPC_GRAY])
ax.bar_label(bars, fmt="%.1f", padding=3)
ax.set_title(f"Gasto medio por edad (p = {comp.loc[0,'p_valor']:.4f})")
ax.set_ylabel("Puntaje de gasto medio"); ax.set_ylim(0, 70)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_comparacion_gasto_edad.png"), bbox_inches="tight")
plt.show()

# R2) Retención gate_30 vs gate_40 en día 1 y día 7 (resultado del A/B).
fig, ax = plt.subplots(figsize=(6.6, 4))
x = np.arange(len(abx)); ancho = 0.36
b1 = ax.bar(x - ancho/2, abx["gate30_tasa"] * 100, ancho, label="gate_30 (control)", color=UPC_GRAY)
b2 = ax.bar(x + ancho/2, abx["gate40_tasa"] * 100, ancho, label="gate_40 (tratamiento)", color=UPC_RED)
ax.bar_label(b1, fmt="%.1f%%", padding=2, fontsize=9); ax.bar_label(b2, fmt="%.1f%%", padding=2, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["Día 1", "Día 7"]); ax.set_ylabel("Retención (%)")
ax.set_title("Retención por grupo del experimento"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_ab_retenciones.png"), bbox_inches="tight")
plt.show()

# R3) Lift (%) por métrica, con marca de significancia.
fig, ax = plt.subplots(figsize=(5.6, 3.8))
colores = [UPC_RED if p < 0.05 else UPC_GRAY for p in abx["p_valor"]]
bars = ax.bar(["Día 1", "Día 7"], abx["lift_pct"], color=colores)
ax.bar_label(bars, fmt="%+.1f%%", padding=3)
ax.axhline(0, color=UPC_INK, linewidth=0.8)
ax.set_title("Lift de gate_40 vs gate_30 (rojo = p < 0.05)")
ax.set_ylabel("Lift (%)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "res_ab_lift.png"), bbox_inches="tight")
plt.show()
print("Figuras de resultados guardadas en:", FIG)

---
## e) Cierre: recomendación de negocio

**Recomendación.** **No mover la puerta al nivel 40.** En la métrica que rige la
decisión -la **retención a 7 días**- el cambio **reduce** la retención en torno a
un **4% relativo**, con una diferencia **estadísticamente significativa**
(p ~ 0.0016) e **intervalo enteramente desfavorable**. Mantener la puerta en el
**nivel 30** protege la base de jugadores activos. Si el equipo quisiera insistir
con otra variante, debería diseñar un **nuevo experimento** con tamaño y duración
fijados de antemano.

**Cómo se llegó (recorrido CRISP-DM de la sesión):**
1. *Negocio* -> pregunta: conviene mover la puerta? (fase 1)
2. *Datos + EDA* -> se entendieron y depuraron los datos (fases 2-3, bloques *a*, *b*).
3. *Análisis* -> se compararon grupos y se corrió el A/B (fase 4, bloques *c*, *d*).
4. *Evaluación* -> significancia estadística **y** práctica (fase 5).
5. *Despliegue* -> recomendación accionable (fase 6, este cierre).

### Entregable de la sesión
Ver `evaluacion/entregable.docx`: un **análisis EDA + recomendación de un A/B** con la
plantilla `plantillas/plantilla_exploracion.docx` y la `plantillas/guia_ab.docx`;
se califica con rúbrica **vigesimal (total = 20)**. Los ejercicios de práctica están
en `evaluacion/drills.docx`. Al cierre se aplica un **control corto**.

### Proyecto integrador
Esta sesión **arranca** el proyecto integrador: completar la
`plantillas/ficha_proyecto_crispdm.docx` con un problema de negocio propio y mapear
sus actividades sobre las 6 fases de CRISP-DM.

### Drills (enunciados; se resuelven en `evaluacion/drills.docx`)
1. **Ubicar fases CRISP-DM:** dado un caso de negocio, asignar cada actividad a una de las 6 fases.
2. **Elegir el gráfico adecuado** para tres preguntas de negocio distintas.
3. **Interpretar un A/B** y recomendar una decisión (mover / no mover / seguir midiendo).

### Para seguir explorando (fuentes de actualidad)
Casos y estudios recientes sobre experimentacion y A/B testing, con enlaces
verificados, en las fuentes de actualidad de la sesión. Ejemplos: como escalan la
experimentacion Booking.com, Netflix y Microsoft; y evidencia empirica sobre
*p-hacking* en el A/B testing de e-commerce.

### Bibliografia (lecturas de la sesion)
- Provost, F. & Fawcett, T. (2013). *Data Science for Business*, cap. 1-2. O'Reilly.
- James, Witten, Hastie & Tibshirani. *An Introduction to Statistical Learning* (ISLR), cap. 2. [statlearning.com](https://www.statlearning.com/)
- Chapman, P. et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*.
- Kohavi, R., Tang, D. & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*. Cambridge University Press.
- Wasserstein, R. & Lazar, N. (2016). *The ASA Statement on p-Values*. *The American Statistician* 70(2).

---
*Cuaderno de la Sesion EPE E1 - Herramientas de Ciencias de Datos - UPC. Datos: Mall
Customers (Kaggle, mirror abierto) y Cookie Cats (Kaggle, mirror abierto). Todas las
cifras provienen de la ejecucion de este cuaderno y de
`resultados/E01_resultados.xlsx`.*